In [ ]:
### Merge single-slice RDS objects and perform batch-corrected clustering using BayesSpace.

In [ ]:
### library
suppressMessages({
    library(Seurat)
    library(data.table)
    library(ggplot2)
    library("stringr")
    library("dplyr")
    library(ggrepel)
    library(SingleCellExperiment)
    library(BayesSpace)
    library(RColorBrewer)
    library(reshape2)  
    library(future)
    library(pheatmap)
    library(cowplot)
    library(harmony)
    library(ggpubr)
    library(viridis)
    library(ggridges)
    require(data.table)
    set.seed(1234)
    library(RColorBrewer)
    cols = c(brewer.pal(9, "Set1"),brewer.pal(8,"Set2")[1:8],brewer.pal(12,"Paired")[1:12],brewer.pal(8,"Dark2")[1:8],brewer.pal(8,"Accent"),brewer.pal(12, "Set3"),brewer.pal(9,"Pastel1"),brewer.pal(8,"Pastel2"))
    })

In [ ]:
### read all rds
lst=list()
i=1
j=1
for (section in c("I-1","I-2","I-3","I-4","I-5")){  ##
    dat=readRDS(paste0('/data/',section,".bin200.rds"))
    dat$T_names = section
    dat = RenameCells(dat,new.names = paste0(section, "_",colnames(x = dat[["Spatial"]])))
    dat <- NormalizeData(dat)
    dat <- SCTransform(dat, assay = "Spatial", verbose = FALSE)
    lst[[i]]=dat
    i=i+1  
}
dat.merge=merge(x=lst[[1]],y=lst[2:length(lst)])
dat.merge$cellid = rownames(dat.merge@meta.data)
# head(dat.merge)

In [ ]:
### adjust coord
chips <- c("I-2", "I-3", "I-4", "I-5")
last_max <- max(dat.merge$coor_x[dat.merge$T_names == "I-1"])
                               
for (chip in chips) {
  idx <- dat.merge$T_names == chip
  shift <- last_max + 100
  dat.merge$coor_x[idx] <- dat.merge$coor_x[idx] + shift
  last_max <- max(dat.merge$coor_x[idx])
}

In [ ]:
### check UMAP
ggplot(dat.merge@meta.data,aes(as.numeric(coor_x), as.numeric(coor_y), fill= area))+ geom_tile()+ theme_classic()+ coord_fixed()+ scale_fill_manual(values =cols)+labs(fill="area")

In [ ]:
### renew stereo
coord <- data.frame(stereo_1 = dat.merge$coor_x, stereo_2 = dat.merge$coor_y)
dat.merge[["stereo"]] <- SeuratObject::CreateDimReducObject(embeddings = as.matrix(coord), key = "stereo_",assay = "SCT")
dat.merge@images[length(unique(dat.merge$T_names))+1:length(names(dat.merge@images))] <- NULL

dat.merge@meta.data$row = dat.merge@meta.data$coor_y
dat.merge@meta.data$col = dat.merge@meta.data$coor_x

p = DimPlot(dat.merge,reduction="stereo",group.by = "area",label = TRUE,label.size = 0.6,pt.size = 1.8,raster=FALSE,cols = cols)+coord_fixed()
p

In [ ]:
### pca 
DefaultAssay(dat.merge) <- "SCT"
variablefeatures = do.call(c,lapply(lst,Seurat::VariableFeatures))
VariableFeatures(dat.merge)<- variablefeatures
dat.merge <- RunPCA(dat.merge, assay = "SCT", verbose = FALSE)

In [ ]:
### RunHarmony
dat.merge=RunHarmony(dat.merge,group.by.vars="T_names", max.iter.harmony = 30) %>% 
                RunUMAP(reduction = "harmony", dims = 1:30) %>%
                FindNeighbors(reduction = "harmony", dims = 1:30)%>% 
                FindClusters(resolution =c(0.5))

In [ ]:
### save rds
saveRDS(dat.merge,"/data/Bin200.merge.debatch.rds")

In [ ]:
plan("multicore", workers = 10)
options(future.globals.maxSize = 100 * 1024^3)

In [ ]:
### Bayes clustering
sce <- as.SingleCellExperiment(dat.merge,assay='SCT')
n_clus = c(16)  ### 16-20
meta = dat.merge@meta.data
for(n_clu in n_clus){
    set.seed(149)
    sce <- spatialCluster(sce, use.dimred = "HARMONY", q = n_clu, nrep = 10000,platform="ST")
    #rename spatial.cluster
    meta$spatial.cluster = colData(sce)$spatial.cluster
    spatial.cluster.n_clu=paste0('Bayes',n_clu)
    names(meta)[names(meta)=='spatial.cluster']=spatial.cluster.n_clu
    ## save bayes meta                        
    saveRDS(meta,paste0("/data/Bin200.debatch.bayes",n_clu,".outcome.rds"))
}